# 13장 진짜 LLM에 같은 질문 던지기

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

모델: [HyperCLOVAX SEED 0.5B](https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B) · [Gemma 3 1B](https://huggingface.co/google/gemma-3-1b-pt) (각 모델의 라이선스를 따릅니다) · 이야기 LLM: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 13-1 진짜 LLM 불러오기

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
from modu_llm import load_llm, load_story_llm
llm = load_llm("naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B")
story = load_story_llm()

## 실습 13-2 크기 비교하기

**먼저 예상해 보세요.** 이야기 LLM에는 파라미터가 약 609만 개 있습니다. HyperCLOVAX의 파라미터는 이야기 LLM의 몇 배쯤일까요? 파라미터가 가장 많이 모인 곳은 어디일까요?

In [ ]:
llm.info()
story.info()

## 실습 13-3 네 LLM의 토큰 나누기

**먼저 예상해 보세요.** 토큰 수가 가장 적은 쪽은 어디일까요? 한국 회사가 만든 두 LLM과 여러 언어를 함께 다루는 gpt-oss 가운데 한국어를 더 큰 조각으로 나누는 쪽은요?

In [ ]:
story.compare_tokens("어느 날, 토끼는 숲으로 산책을 갔어요. 토끼는 곰을 만났어요.",
                     "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B",
                     "K-intelligence/Midm-2.0-Mini-Instruct",
                     "openai/gpt-oss-20b")

## 실습 13-4 더 해 보기: 사전에 없는 글자

**먼저 예상해 보세요.** 이야기 LLM은 글자 사전에 없는 '똠'과 웃는 얼굴 그림 글자 😀를 건너뛰고 읽습니다. 진짜 LLM들도 건너뛸까요?

In [ ]:
story.compare_tokens("똠얌꿍 좋아 😀",
                     "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B",
                     "K-intelligence/Midm-2.0-Mini-Instruct",
                     "openai/gpt-oss-20b")

## 실습 13-5 다음 토큰 맞히기

**먼저 예상해 보세요.** 이야기 LLM은 "토끼는 숲으로 산" 다음 글자로 '책'에 99.6%를 주었습니다. 어린이 이야기에서 "숲으로 산" 다음은 거의 언제나 "산책"이었으니까요. 온갖 글을 읽은 HyperCLOVAX도 이렇게 확신할까요?

In [ ]:
llm.next_chars("토끼는 숲으로 산")

## 실습 13-6 이야기 LLM이 모르는 질문

**먼저 예상해 보세요.** 이번에는 두 LLM에게 "대한민국의 수도는" 다음을 물어보겠습니다. 두 LLM의 1위 후보는 같을까요? 이야기 LLM은 글자 단위라서 띄어쓰기까지 넣어 줍니다.

In [ ]:
llm.next_chars("대한민국의 수도는")
story.next_chars("대한민국의 수도는 ")

## 실습 13-7 이어 쓰기

**먼저 예상해 보세요.** 어린이 이야기의 한 문장을 주면, 온갖 글을 읽은 HyperCLOVAX는 어떤 말투로 이어 쓸까요?

In [ ]:
llm.write("토끼는 숲으로 산책을 갔어요.", length=40)

## 실습 13-8 어느 토큰에서 놀랐을까?

**먼저 예상해 보세요.** "토끼가 숲에 갔어요."와 "토끼가 숲에 갑요."는 앞부분이 같고 끝만 다릅니다. HyperCLOVAX가 두 문장에서 가장 크게 놀라는 토큰은 각각 무엇일까요?

In [ ]:
llm.losses("토끼가 숲에 갔어요.")
llm.losses("토끼가 숲에 갑요.")

## 실습 13-9 문장마다 손실 비교하기

**먼저 예상해 보세요.** 다음 네 문장을 두 LLM에 넣으면 손실은 어떻게 나올까요? 둘째 문장은 낱말 순서를, 셋째 문장은 글자 순서를 뒤섞었고, 넷째 문장은 이야기가 아닌 문장입니다.

In [ ]:
sentences = ["어느 날, 토끼는 숲으로 산책을 갔어요.",
             "어느 날, 숲으로 토끼는 갔어요 산책을.",
             "날 어느, 는토끼 로으숲 을책산 요어갔.",
             "대한민국의 수도는 서울입니다."]
llm.compare_losses(sentences)
story.compare_losses(sentences)

## 실습 13-10 같은 문구를 두 번 넣기

**먼저 예상해 보세요.** "파란 우산 노란 장화 빨간 모자 초록 가방."을 두 번 이어 넣으면, HyperCLOVAX의 두 번째 손실은 첫 번째보다 얼마나 작아질까요?

In [ ]:
llm.losses("파란 우산 노란 장화 빨간 모자 초록 가방.", repeat=2)

## 실습 13-11 이야기 LLM도 따라 할까?

**먼저 예상해 보세요.** 이야기 LLM에 같은 문구를 두 번 넣으면 두 번째 손실이 떨어질까요? 이야기 LLM은 문구를 한 글자씩 읽으므로 글자마다 비교합니다.

In [ ]:
story.losses("파란 우산 노란 장화 빨간 모자 초록 가방.", repeat=2)

## 실습 13-12 뜻 없는 낱말 줄로 확인하기

**먼저 예상해 보세요.** "콩 별 숲 달 꿈 눈 빛 꽃"은 처음 보면 다음 낱말을 짐작할 길이 없습니다. 두 번째에는 두 LLM이 각각 어떻게 될까요?

In [ ]:
llm.losses("콩 별 숲 달 꿈 눈 빛 꽃", repeat=2)
story.losses("콩 별 숲 달 꿈 눈 빛 꽃", repeat=2)